# 03 - Evaluation

Runs the 30-question golden set through both models. Retrieval metrics (Context Precision/Recall, MRR) come from the gold pages; generation metrics (Faithfulness, Relevancy, Correctness) come from an LLM judge (gemma3:4b, a different model from both generators).

In [ ]:
%pip install -q ollama chromadb pandas

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

In [ ]:
from src.evaluation import run_evaluation

smol_rows = run_evaluation("smollm")
llama_rows = run_evaluation("llama")

### Compare the two models

In [ ]:
import json, pandas as pd
from src.config import RESULTS_DIR

smol_summary = json.loads((RESULTS_DIR / "smollm_summary.json").read_text())
llama_summary = json.loads((RESULTS_DIR / "llama_summary.json").read_text())
pd.DataFrame({"smollm2-135m": smol_summary, "llama3.2-3b": llama_summary})

### Failure analysis

Retrieval metrics are the same for both models by design (same retriever, same top-k) - so any gap in
Faithfulness, Relevancy or Correctness is a generation problem, not a retrieval problem.

Real examples seen in this run:
- **Hallucination on out-of-scope questions.** Asked for a tuition fee that isn't in the handbook,
  SmolLM2 invented a number; Llama 3.2-3B correctly replied "Not in handbook."
- **Factual errors on simple lookups.** Asked where the university is located, SmolLM2 answered
  "Karaikal, Karnataka" - fabricated, not in the handbook at all. Llama answered correctly ("Karjat,
  Raigad").
- **Refusal rate.** Across the 4 not-in-handbook questions, Llama refused correctly 4/4 times; SmolLM2
  refused correctly 0/4 times.

At 135M parameters, SmolLM2 has not learned reliable "I don't know" behaviour or careful fact recall,
even when the correct context was retrieved for both models identically.